In [4]:
import sys
import os

sys.path.append(os.path.abspath(".."))

from datetime import datetime, timedelta

from core.portfolio.backtest import BacktestEngine
from core.portfolio.asset import Portfolio
from core.data.instrument.provider import InstrumentProvider
from core.data.instrument.instrument import Instrument
from core.ml.pricedir import PriceDirPredictorRF

ip = InstrumentProvider()
instrument: Instrument = ip.get_instrument('MSFT')

HORIZON = 5                     # Prediction horizon (days)
RISK_PER_TRADE_PCT = 0.02       # Max portfolio risk per trade (2%)
MAX_PORTFOLIO_CAP_PCT = 0.40    # Hard cap on single position allocation (40% max cash)

predictor = PriceDirPredictorRF(instrument=instrument, horizon_days=HORIZON)

portfolio = Portfolio('BT1', 'USD')
start_date = datetime(2025, 1, 1)
engine = BacktestEngine(portfolio, start_date=start_date, initial_cash=1000.0)

days_since_last_train = HORIZON  

def atr_risk_sized_ml_strategy():
    global days_since_last_train

    current_date = engine.date

    if days_since_last_train >= HORIZON:
        training_cutoff = current_date - timedelta(days=HORIZON)
        train_stats = predictor.train(end_date=training_cutoff)
        print(f"\n[RETRAINED {current_date.strftime('%Y-%m-%d')}] "
              f"Cutoff: {training_cutoff.strftime('%Y-%m-%d')} | "
              f"Acc: {train_stats['train_accuracy']:.2%}")
        days_since_last_train = 0

    forecast = predictor.predict_at_date(as_of_date=current_date)
    signal = forecast['signal']
    confidence = forecast['confidence_up']
    atr = forecast['atr']
    close_price = forecast['close']

    has_position = any(a.symbol == instrument.symbol and a.volume > 0 for a in portfolio.assets)

    if signal == 1 and confidence > 0.55 and not has_position:
        if atr > 0 and close_price > 0:
            total_portfolio_value = engine.total_value
            risk_budget = total_portfolio_value * RISK_PER_TRADE_PCT
            atr_stop_distance = 2.0 * atr

            target_shares = risk_budget / atr_stop_distance

            calculated_amount = target_shares * close_price

            max_cash_allowed = engine.cash * MAX_PORTFOLIO_CAP_PCT
            final_allocation = min(calculated_amount, max_cash_allowed)

            if final_allocation > 50:
                engine.buy(instrument, amount=final_allocation)
                print(
                    f"  [{current_date.strftime('%Y-%m-%d')}] BUY {instrument.symbol} | "
                    f"Allocated: ${final_allocation:,.2f} | "
                    f"Conf: {confidence:.2%} | ATR: ${atr:.2f}"
                )

    elif signal == 0 and has_position:
        engine.sell(symbol=instrument.symbol, amount=engine.cash)
        print(f"  [{current_date.strftime('%Y-%m-%d')}] SELL Executed | Conf DOWN: {forecast['confidence_down']:.2%}")

    days_since_last_train += 1

print("--- Starting Horizon-Based Walk-Forward Backtest ---")
for i in range(500):
    engine.next_day(strategy=atr_risk_sized_ml_strategy)
    print(f"Day {i+1} ({engine.date.strftime('%Y-%m-%d')}): Assets Value: ${engine.assets_values:,.2f}, Cash: ${engine.cash}, Total Value: ${engine.total_value}")


--- Starting Horizon-Based Walk-Forward Backtest ---

[RETRAINED 2025-01-01] Cutoff: 2024-12-27 | Acc: 62.88%
Day 1 (2025-01-02): Assets Value: $0.00, Cash: $1000.0, Total Value: $1000.0
Day 2 (2025-01-03): Assets Value: $0.00, Cash: $1000.0, Total Value: $1000.0
Day 3 (2025-01-04): Assets Value: $0.00, Cash: $1000.0, Total Value: $1000.0
Day 4 (2025-01-05): Assets Value: $0.00, Cash: $1000.0, Total Value: $1000.0
Day 5 (2025-01-06): Assets Value: $0.00, Cash: $1000.0, Total Value: $1000.0

[RETRAINED 2025-01-06] Cutoff: 2025-01-01 | Acc: 62.93%
Day 6 (2025-01-07): Assets Value: $0.00, Cash: $1000.0, Total Value: $1000.0
Day 7 (2025-01-08): Assets Value: $0.00, Cash: $1000.0, Total Value: $1000.0
Day 8 (2025-01-09): Assets Value: $0.00, Cash: $1000.0, Total Value: $1000.0
Day 9 (2025-01-10): Assets Value: $0.00, Cash: $1000.0, Total Value: $1000.0
Day 10 (2025-01-11): Assets Value: $0.00, Cash: $1000.0, Total Value: $1000.0

[RETRAINED 2025-01-11] Cutoff: 2025-01-06 | Acc: 63.19%
Day 1